# Preprocessing

Giai đoạn tiền xử lý sau EDA. Chạy lại trên bản snapshot dẫn xuất; giữ nguyên nguồn và pipeline collaborator.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project_directory = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                         if (path / "recommendation/config.json").is_file())
sys.path.insert(0, str(project_directory / "recommendation/src"))
from pipeline_common import ROOT, ARTIFACTS, REPORTS, read_json, load_csv


In [2]:
from preprocess import build_preprocessing
manifest = build_preprocessing()
display(pd.Series({key:value for key,value in manifest.items() if key not in ["files", "scenarios"]}))


version                                                            four_methods_v1
protocol_sha256                  a26176f96f3e69f562844805f1e804d5d89260ce12e92e...
source_manifest_sha256           0f655004e46f5e1fd2dc8a335017ae1e722bc18db3ffaf...
gold_events                                                                  28762
historical_clean_events                                                      28255
all_clean_events                                                             57017
catalogue_videos                                                               560
completion_mismatches_flagged                                                    0
historical_identity_verified                                                 False
dtype: object

## Quy tắc và nhãn

Loại thiếu ID, bot, watch âm/quá 3.600 giây và event ID lặp; giữ null watch đi cùng block. Positive yêu cầu không block/skip và completion ≥0,8. Không cắt completion >1. Cặp lần đầu dùng cho video mới; lượt xem lặp vẫn lưu riêng.

In [3]:
display(load_csv(ARTIFACTS / "preprocessing/cleaning_audit.csv"))
for scenario, details in manifest["scenarios"].items():
    print(scenario)
    display(pd.DataFrame(details["counts"]).T)


,origin,rule,removed,remaining
0,current,missing_identifier,0,28762
1,current,bot,0,28762
2,current,invalid_watch_time,0,28762
3,current,invalid_action_flag,0,28762
4,current,invalid_timestamp_delay,0,28762
5,current,missing_watch_without_block,0,28762
6,current,duplicate_event_id,0,28762
7,historical,missing_identifier,519,30277
8,historical,bot,530,29747
9,historical,invalid_watch_time,995,28752


primary


,test,train,validation
historical_assumed_single,4009,18707,4009
run=2ad86635f1a18e06,697,3249,696
run=94f1d592a7787780,510,2377,510
run=e1d06d3cf6587c44,896,4179,895


historical_day


,test,train,validation
historical_assumed_single,4121,19231,4121
run=2ad86635f1a18e06,697,3249,696
run=94f1d592a7787780,510,2377,510
run=e1d06d3cf6587c44,896,4179,895


## Kết luận

Dữ liệu có provenance và checksum. Train/validation/test chia theo thời gian toàn cohort. Lịch sử vẫn có giả định danh tính; chưa huấn luyện trong notebook này.